In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-04-16 13:48:09.425809: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776347289.650424      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776347289.717533      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776347290.233281      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776347290.233367      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776347290.233370      22 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train.npy'))
X_val = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val.npy'))
X_test = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test.npy'))

In [4]:
X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


In [5]:
import sys
MODULE_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-utils"
if MODULE_PATH not in sys.path:
    sys.path.append(MODULE_PATH)

from model_utils import (
    create_bilstm,
    get_callbacks,
    find_best_threshold,
    full_evaluation,
)

In [6]:
from sklearn.utils.class_weight import compute_class_weight
import tensorflow.keras.backend as K
import numpy as np
import tensorflow as tf

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}
print("Trọng số lớp Train:", class_weights_dict)

lstm_units_options = [16, 32, 50]
dropout_rate_options = [0.3, 0.4, 0.5]
batch_size_options = [256, 512]
num_epochs = 15

results = []

for n_units in lstm_units_options:
    for dropout_rate in dropout_rate_options:
        for batch_size in batch_size_options:
            print(f"\nĐang test: units={n_units}, dropout={dropout_rate}, batch_size={batch_size}")

            K.clear_session()

            model = create_bilstm(
                n_units=n_units,
                dropout=dropout_rate,
                seq_len=X_train.shape[1],
                n_features=X_train.shape[2],
                lr=1e-3
            )

            history = model.fit(
                X_train, y_train,
                epochs=num_epochs,
                batch_size=batch_size,
                class_weight=class_weights_dict,
                validation_data=(X_val, y_val),
                shuffle=True,
                verbose=0
            )

            val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision = model.evaluate(
                X_val, y_val, verbose=0
            )

            print(
                f"Val AUROC: {val_auroc:.4f} | "
                f"Val AUPRC: {val_auprc:.4f} | "
                f"Recall: {val_recall:.4f} | "
                f"Precision: {val_precision:.4f}"
            )

            results.append((
                n_units, dropout_rate, batch_size,
                val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision
            ))

# Chọn theo AUPRC thay vì AUROC
best_result = sorted(results, key=lambda x: x[6], reverse=True)[0]

print("\n" + "="*60)
print("Best Hyperparameters:")
print(f"- LSTM Units:   {best_result[0]}")
print(f"- Dropout Rate: {best_result[1]}")
print(f"- Batch Size:   {best_result[2]}")
print(f"- Val AUROC:    {best_result[5]:.4f}")
print(f"- Val AUPRC:    {best_result[6]:.4f}")
print("="*60)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}

Đang test: units=16, dropout=0.3, batch_size=256


I0000 00:00:1776347334.765266      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776347334.771683      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1776347344.022590      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


Val AUROC: 0.7980 | Val AUPRC: 0.3255 | Recall: 0.6137 | Precision: 0.2757

Đang test: units=16, dropout=0.3, batch_size=512
Val AUROC: 0.7936 | Val AUPRC: 0.3201 | Recall: 0.5890 | Precision: 0.2784

Đang test: units=16, dropout=0.4, batch_size=256
Val AUROC: 0.7948 | Val AUPRC: 0.3481 | Recall: 0.5840 | Precision: 0.3067

Đang test: units=16, dropout=0.4, batch_size=512
Val AUROC: 0.7945 | Val AUPRC: 0.3520 | Recall: 0.5847 | Precision: 0.2944

Đang test: units=16, dropout=0.5, batch_size=256
Val AUROC: 0.7827 | Val AUPRC: 0.3083 | Recall: 0.5883 | Precision: 0.2799

Đang test: units=16, dropout=0.5, batch_size=512
Val AUROC: 0.7931 | Val AUPRC: 0.3363 | Recall: 0.5929 | Precision: 0.2846

Đang test: units=32, dropout=0.3, batch_size=256
Val AUROC: 0.7757 | Val AUPRC: 0.3191 | Recall: 0.4869 | Precision: 0.3283

Đang test: units=32, dropout=0.3, batch_size=512
Val AUROC: 0.7623 | Val AUPRC: 0.3102 | Recall: 0.4751 | Precision: 0.3285

Đang test: units=32, dropout=0.4, batch_size=256


In [7]:
import tensorflow.keras.backend as K

K.clear_session()

final_model = create_bilstm(
    n_units=best_result[0],
    dropout=best_result[1],
    seq_len=X_train.shape[1],
    n_features=X_train.shape[2],
    lr=1e-3
)

callbacks = get_callbacks(
    checkpoint_path='best_lstm_model.keras',
    monitor='val_auprc'
)

print("🚀 ĐANG HUẤN LUYỆN FINAL MODEL...")
history = final_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=best_result[2],
    class_weight=class_weights_dict,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

🚀 ĐANG HUẤN LUYỆN FINAL MODEL...
Epoch 1/50
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6338 - auprc: 0.1621 - auroc: 0.6903 - loss: 0.6416 - precision: 0.1266 - recall: 0.6453
Epoch 1: val_auprc improved from -inf to 0.29214, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.6339 - auprc: 0.1623 - auroc: 0.6905 - loss: 0.6414 - precision: 0.1267 - recall: 0.6455 - val_accuracy: 0.7720 - val_auprc: 0.2921 - val_auroc: 0.8295 - val_loss: 0.4497 - val_precision: 0.2158 - val_recall: 0.7547 - learning_rate: 0.0010
Epoch 2/50
283/285 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7385 - auprc: 0.2796 - auroc: 0.8390 - loss: 0.4970 - precision: 0.1971 - recall: 0.8120
Epoch 2: val_auprc improved from 0.29214 to 0.31659, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.7387 - auprc: 0.2798 - auroc: 0.8391 - loss: 0.4968 - precision: 0.1972 - recall: 0.8121 - val_accuracy: 0.7856 - val_auprc

In [8]:
val_prob = final_model.predict(X_val, verbose=0).ravel()

best_val_threshold = find_best_threshold(
    y_true=y_val,
    y_prob=val_prob,
    min_sensitivity=0.80,
    threshold_range=(0.05, 0.95),
    step=0.01
)

print("Best threshold from validation:")
print(best_val_threshold)

Best threshold from validation:
{'threshold': 0.33, 'accuracy': 0.7597344044593819, 'sensitivity': np.float64(0.8009308986752596), 'specificity': np.float64(0.7563306117619216), 'precision': 0.21357647508115332, 'recall': 0.8009308986752596, 'f1': 0.3372277078465365, 'youden_j': np.float64(0.5572615104371812), 'tn': 25567, 'fp': 8237, 'fn': 556, 'tp': 2237}


In [9]:
test_prob = final_model.predict(X_test, verbose=0).ravel()

full_evaluation(
    y_true=y_test,
    y_prob=test_prob,
    threshold=best_val_threshold["threshold"],
    label="TEST"
)

df_lstm = pd.DataFrame({
    'y_true': y_test,
    'pred_lstm': test_prob,
    'pred_label': (test_prob >= best_val_threshold["threshold"]).astype(int)
})
df_lstm.to_csv('predictions_lstm.csv', index=False)

print("✅ Đã xuất 'predictions_lstm.csv'")


  TEST — threshold = 0.33
  AUROC        : 0.8690
  AUPRC        : 0.3594
  Accuracy     : 0.7652
  Sensitivity  : 0.8280
  Specificity  : 0.7604
  Precision    : 0.2079
  Recall       : 0.8280
  F1-score     : 0.3324
  Youden's J   : 0.5884
  Confusion    : TN=32195 FP=10142 FN=553 TP=2662

✅ Đã xuất 'predictions_lstm.csv'
